# Hartree-Fock SCF Calculation

This notebook implements a complete **Restricted Hartree-Fock** (RHF) self-consistent
field calculation for water using LibAccInt for all molecular integrals.

**What you'll learn:**
- The Roothaan-Hall equations and SCF iteration
- Symmetric orthogonalization ($X = S^{-1/2}$)
- Core Hamiltonian initial guess
- Fock matrix construction with `FockBuilder`
- DIIS acceleration for faster convergence
- Effect of basis set on total energy

In [ ]:
import numpy as np
from scipy import linalg
import libaccint as lai

print(f"LibAccInt {lai.__version__}")
print(f"CUDA: {lai.has_cuda_backend()}")

## 1. The Hartree-Fock Method

The Roothaan-Hall equation is a matrix eigenvalue problem:

$$\mathbf{F} \mathbf{C} = \mathbf{S} \mathbf{C} \boldsymbol{\varepsilon}$$

where $\mathbf{F}$ is the Fock matrix, $\mathbf{C}$ the MO coefficients, $\mathbf{S}$ the
overlap matrix, and $\boldsymbol{\varepsilon}$ the orbital energies.

Since $\mathbf{F}$ depends on $\mathbf{C}$ (through the density matrix $\mathbf{D}$),
this must be solved iteratively until self-consistency.

In [ ]:
# Water molecule in Bohr
atoms = [
    lai.Atom(8, [0.0, 0.0, 0.0]),                  # O
    lai.Atom(1, [0.0,  1.43233673, -1.10866041]),   # H
    lai.Atom(1, [0.0, -1.43233673, -1.10866041]),   # H
]

basis = lai.basis_set("6-31G*", atoms)
engine = lai.Engine(basis)
nbf = basis.n_basis_functions()
n_elec = 10
n_occ = n_elec // 2

# The engine transparently uses GPU when available (BackendHint.Auto).
# PreferGPU is used below for the expensive 2e step; it falls back to CPU
# if no GPU backend was compiled in.
print(f"H\u2082O / 6-31G*: {nbf} basis functions, {n_occ} occupied orbitals")
print(f"GPU available: {engine.gpu_available()}")

## 2. Nuclear Repulsion Energy

$$E_{\text{nuc}} = \sum_{A<B} \frac{Z_A Z_B}{|\mathbf{R}_A - \mathbf{R}_B|}$$

In [ ]:
def compute_nuclear_repulsion(atoms):
    E_nuc = 0.0
    for i in range(len(atoms)):
        for j in range(i + 1, len(atoms)):
            ri, rj = atoms[i].position, atoms[j].position
            r = np.sqrt((ri.x-rj.x)**2 + (ri.y-rj.y)**2 + (ri.z-rj.z)**2)
            E_nuc += atoms[i].atomic_number * atoms[j].atomic_number / r
    return E_nuc

E_nuc = compute_nuclear_repulsion(atoms)
print(f"Nuclear repulsion energy: {E_nuc:.12f} Hartree")

# Verify against known reference
E_nuc_ref = 9.182637358503053
print(f"Reference:               {E_nuc_ref:.12f}")
print(f"Match: {abs(E_nuc - E_nuc_ref) < 1e-10}")

## 3. One-Electron Integrals

In [ ]:
S = engine.compute_overlap_matrix()
T = engine.compute_kinetic_matrix()
V = engine.compute_nuclear_matrix(atoms)
H_core = T + V

print(f"S, T, V, H_core: all ({nbf}, {nbf})")

## 4. Symmetric Orthogonalization

We transform to an orthogonal basis using $\mathbf{X} = \mathbf{S}^{-1/2}$,
computed via eigendecomposition of $\mathbf{S}$:

$$\mathbf{S} = \mathbf{U} \mathbf{s} \mathbf{U}^T \quad\Rightarrow\quad \mathbf{X} = \mathbf{U} \mathbf{s}^{-1/2} \mathbf{U}^T$$

In [ ]:
s_vals, U = linalg.eigh(S)

# Discard near-zero eigenvalues for numerical stability
mask = s_vals > 1e-10
s_inv_sqrt = np.zeros_like(s_vals)
s_inv_sqrt[mask] = 1.0 / np.sqrt(s_vals[mask])

X = U @ np.diag(s_inv_sqrt) @ U.T

# Verify: X^T S X = I
I_check = X.T @ S @ X
print(f"X^T S X = I: {np.allclose(I_check, np.eye(nbf))}")
print(f"Max |X^T S X - I|: {np.max(np.abs(I_check - np.eye(nbf))):.2e}")

## 5. Initial Guess

Diagonalize $H_{\text{core}}$ in the orthogonal basis for the initial MO coefficients.

In [ ]:
# Transform H_core to orthogonal basis
H_prime = X.T @ H_core @ X

# Diagonalize
eps, C_prime = linalg.eigh(H_prime)

# Back-transform to AO basis
C = X @ C_prime

# Build initial density: D = 2 * C_occ @ C_occ^T
C_occ = C[:, :n_occ]
D = 2.0 * C_occ @ C_occ.T

print(f"Initial orbital energies: {eps[:n_occ+2]}")
print(f"HOMO-LUMO gap: {eps[n_occ] - eps[n_occ-1]:.4f} Hartree")

## 6. SCF Iteration (Basic)

Each iteration:
1. Build Fock matrix $\mathbf{F} = \mathbf{H} + \mathbf{J} - \frac{1}{2}\mathbf{K}$ via `engine.compute(op, fock_builder)`
2. Transform: $\mathbf{F}' = \mathbf{X}^T \mathbf{F} \mathbf{X}$
3. Diagonalize $\mathbf{F}'$ to get new orbitals
4. Build new density and check convergence

In [ ]:
E_old = 0.0
max_iter = 100
e_thresh = 1e-10
d_thresh = 1e-8

# Use PreferGPU for the expensive 2e step — auto-falls back to CPU if unavailable
hint = lai.BackendHint.PreferGPU

print(f"{'Iter':>4} {'E_total':>20} {'dE':>12} {'dD_max':>12}")
print("-" * 52)

for iteration in range(max_iter):
    # Build Fock matrix via engine.compute(op, consumer, hint)
    D_c = np.ascontiguousarray(D, dtype=np.float64)
    fock = lai.FockBuilder(nbf)
    fock.set_density(D_c)
    engine.compute(lai.Operator.coulomb(), fock, hint)

    J = fock.get_coulomb_matrix()
    K = fock.get_exchange_matrix()
    F = H_core + J - 0.5 * K

    # Electronic energy: E_elec = 0.5 * Tr[(H_core + F) * D]
    E_elec = 0.5 * np.sum((H_core + F) * D)
    E_total = E_elec + E_nuc

    # Diagonalize in orthogonal basis
    F_prime = X.T @ F @ X
    eps, C_prime = linalg.eigh(F_prime)
    C = X @ C_prime

    # New density
    C_occ = C[:, :n_occ]
    D_new = 2.0 * C_occ @ C_occ.T

    # Convergence
    dE = abs(E_total - E_old)
    dD = np.max(np.abs(D_new - D))

    print(f"{iteration:>4} {E_total:>20.12f} {dE:>12.2e} {dD:>12.2e}")

    if dE < e_thresh and dD < d_thresh and iteration > 0:
        print(f"\nConverged in {iteration + 1} iterations!")
        break

    D = D_new
    E_old = E_total
else:
    print("\nWARNING: SCF did not converge!")

E_basic = E_total
print(f"\nTotal energy: {E_total:.12f} Hartree")

## 7. DIIS Acceleration

Direct Inversion in the Iterative Subspace (DIIS) dramatically speeds convergence
by extrapolating from a history of Fock matrices and error vectors.

The DIIS error vector is $\mathbf{e} = \mathbf{F}\mathbf{D}\mathbf{S} - \mathbf{S}\mathbf{D}\mathbf{F}$
(which vanishes at self-consistency).

In [ ]:
class DIIS:
    """Simple DIIS extrapolation."""
    def __init__(self, max_size=6):
        self.max_size = max_size
        self.fock_list = []
        self.error_list = []

    def add(self, F, error):
        self.fock_list.append(F.copy())
        self.error_list.append(error.copy())
        if len(self.fock_list) > self.max_size:
            self.fock_list.pop(0)
            self.error_list.pop(0)

    def extrapolate(self):
        n = len(self.fock_list)
        if n < 2:
            return self.fock_list[-1]

        # Build B matrix: B_ij = <e_i | e_j>
        B = np.zeros((n + 1, n + 1))
        for i in range(n):
            for j in range(n):
                B[i, j] = np.sum(self.error_list[i] * self.error_list[j])
        # Lagrange constraint row/column
        B[n, :n] = -1.0
        B[:n, n] = -1.0

        rhs = np.zeros(n + 1)
        rhs[n] = -1.0

        c = np.linalg.solve(B, rhs)
        return sum(c[i] * self.fock_list[i] for i in range(n))

In [ ]:
# Re-run SCF with DIIS
# Reset to core Hamiltonian guess
H_prime = X.T @ H_core @ X
eps, C_prime = linalg.eigh(H_prime)
C = X @ C_prime
D = 2.0 * C[:, :n_occ] @ C[:, :n_occ].T
E_old = 0.0
diis = DIIS(max_size=6)
hint = lai.BackendHint.PreferGPU  # GPU when available, CPU fallback

print(f"{'Iter':>4} {'E_total':>20} {'dE':>12} {'dD_max':>12}")
print("-" * 52)

for iteration in range(max_iter):
    D_c = np.ascontiguousarray(D, dtype=np.float64)
    fock = lai.FockBuilder(nbf)
    fock.set_density(D_c)
    engine.compute(lai.Operator.coulomb(), fock, hint)

    J = fock.get_coulomb_matrix()
    K = fock.get_exchange_matrix()
    F = H_core + J - 0.5 * K

    E_elec = 0.5 * np.sum((H_core + F) * D)
    E_total = E_elec + E_nuc

    # DIIS error: e = FDS - SDF
    error = F @ D @ S - S @ D @ F
    diis.add(F, error)
    F_diis = diis.extrapolate()

    # Diagonalize DIIS-extrapolated Fock matrix
    F_prime = X.T @ F_diis @ X
    eps, C_prime = linalg.eigh(F_prime)
    C = X @ C_prime

    D_new = 2.0 * C[:, :n_occ] @ C[:, :n_occ].T

    dE = abs(E_total - E_old)
    dD = np.max(np.abs(D_new - D))

    print(f"{iteration:>4} {E_total:>20.12f} {dE:>12.2e} {dD:>12.2e}")

    if dE < 1e-12 and dD < 1e-10 and iteration > 0:
        # Recompute final energy with converged density
        D = D_new
        D_c = np.ascontiguousarray(D, dtype=np.float64)
        fock_final = lai.FockBuilder(nbf)
        fock_final.set_density(D_c)
        engine.compute(lai.Operator.coulomb(), fock_final, hint)
        J_f = fock_final.get_coulomb_matrix()
        K_f = fock_final.get_exchange_matrix()
        F_f = H_core + J_f - 0.5 * K_f
        E_elec = 0.5 * np.sum((H_core + F_f) * D)
        E_total = E_elec + E_nuc
        print(f"\nConverged in {iteration + 1} iterations!")
        break

    D = D_new
    E_old = E_total
else:
    print("\nWARNING: SCF did not converge!")

print(f"\nFinal energy: {E_total:.12f} Hartree")

## 8. Results

In [ ]:
print(f"H\u2082O / 6-31G* RHF Results")
print(f"{'='*40}")
print(f"Nuclear repulsion: {E_nuc:>18.12f} Ha")
print(f"Electronic energy: {E_elec:>18.12f} Ha")
print(f"Total energy:      {E_total:>18.12f} Ha")
print(f"\nOrbital energies (Hartree):")
for i, e in enumerate(eps):
    occ = "occ" if i < n_occ else "vir"
    print(f"  {i:>3d} ({occ}): {e:>12.6f}")

## 9. Basis Set Comparison

Run the same SCF with different basis sets to see how basis quality affects the total energy.

In [ ]:
def run_rhf(atoms, basis_name, n_electrons, hint=lai.BackendHint.PreferGPU):
    """Run RHF SCF and return total energy. Uses GPU when available."""
    b = lai.basis_set(basis_name, atoms)
    eng = lai.Engine(b)
    n = b.n_basis_functions()
    nocc = n_electrons // 2

    S_ = eng.compute_overlap_matrix()
    H_ = eng.compute_core_hamiltonian(atoms)
    E_nuc_ = compute_nuclear_repulsion(atoms)

    # Orthogonalization
    sv, Uv = linalg.eigh(S_)
    si = np.zeros_like(sv)
    si[sv > 1e-10] = 1.0 / np.sqrt(sv[sv > 1e-10])
    X_ = Uv @ np.diag(si) @ Uv.T

    # Initial guess
    ep, Cp = linalg.eigh(X_.T @ H_ @ X_)
    C_ = X_ @ Cp
    D_ = 2.0 * C_[:, :nocc] @ C_[:, :nocc].T

    E_old_ = 0.0
    diis_ = DIIS(6)

    for it in range(100):
        Dc = np.ascontiguousarray(D_, dtype=np.float64)
        fb = lai.FockBuilder(n)
        fb.set_density(Dc)
        eng.compute(lai.Operator.coulomb(), fb, hint)
        F_ = H_ + fb.get_coulomb_matrix() - 0.5 * fb.get_exchange_matrix()

        E_el = 0.5 * np.sum((H_ + F_) * D_)
        E_tot = E_el + E_nuc_

        err = F_ @ D_ @ S_ - S_ @ D_ @ F_
        diis_.add(F_, err)
        F_d = diis_.extrapolate()

        ep, Cp = linalg.eigh(X_.T @ F_d @ X_)
        C_ = X_ @ Cp
        D_new = 2.0 * C_[:, :nocc] @ C_[:, :nocc].T

        dE = abs(E_tot - E_old_)
        dD = np.max(np.abs(D_new - D_))

        if dE < 1e-12 and dD < 1e-10 and it > 0:
            # Final energy with converged density
            D_ = D_new
            Dc = np.ascontiguousarray(D_, dtype=np.float64)
            fb2 = lai.FockBuilder(n)
            fb2.set_density(Dc)
            eng.compute(lai.Operator.coulomb(), fb2, hint)
            F_f = H_ + fb2.get_coulomb_matrix() - 0.5 * fb2.get_exchange_matrix()
            E_el = 0.5 * np.sum((H_ + F_f) * D_)
            E_tot = E_el + E_nuc_
            return E_tot, it + 1, n

        D_ = D_new
        E_old_ = E_tot

    return E_tot, 100, n  # not converged

In [ ]:
# Reference: PySCF, same geometry, cart=True, conv_tol=1e-12
references = {
    "STO-3G": -74.963110239453101,
    "6-31G":  -75.983978013076069,
}

print(f"{'Basis':<12} {'nbf':>5} {'Iters':>6} {'E_total (Ha)':>22} {'vs ref':>12}")
print("-" * 62)

for name in ["STO-3G", "6-31G", "6-31G*", "cc-pVDZ"]:
    E, niter, n = run_rhf(atoms, name, n_elec)
    ref_str = ""
    if name in references:
        ref_str = f"{abs(E - references[name]):.2e}"
    print(f"{name:<12} {n:>5} {niter:>6} {E:>22.12f} {ref_str:>12}")

print("\nEnergies decrease (more negative) with larger basis sets,")
print("approaching the Hartree-Fock limit.")

In [ ]:
# Final verification: STO-3G must match PySCF reference to 1e-10
E_sto3g, _, _ = run_rhf(atoms, "STO-3G", n_elec)
ref = -74.963110239453101
err = abs(E_sto3g - ref)
print(f"STO-3G energy: {E_sto3g:.12f}")
print(f"PySCF ref:     {ref:.12f}")
print(f"Error:         {err:.2e}")
assert err < 1e-8, f"STO-3G energy error {err} exceeds tolerance!"
print("PASSED: Energy matches PySCF reference.")

## Summary

We implemented a complete RHF SCF calculation using only LibAccInt for integrals
and NumPy/SciPy for linear algebra. Key steps:

1. **One-electron integrals** via `compute_overlap_matrix()`, `compute_core_hamiltonian()`
2. **Symmetric orthogonalization** $X = S^{-1/2}$ via `scipy.linalg.eigh`
3. **Fock matrix** via `FockBuilder` + `engine.compute(op, fock_builder, hint)`
4. **DIIS acceleration** for rapid convergence
5. **Basis set comparison** showing convergence toward the HF limit

The converged STO-3G energy matches PySCF to machine precision, validating both
the integral library and the SCF implementation.